# Append the 1,219 older / duplicate photo posts

**Run this notebook AFTER the original notebook finishes 2,573/2,573.**

Destination: **Bollywood Movie — -1004492918137**.

This second pass uploads all 1,219 previously excluded photo posts with their own
original captions and links. It does not deduplicate them again. The final total is
**3,792 available posts** (2,573 + 1,219). The 29 missing-video posts stay excluded.
These older posts will appear AFTER the first batch, with new upload timestamps.

The first-batch list and progress file are preserved. A separate extra-batch progress
file lets you resume without restarting either batch. You must use the same Google
Drive account and the same Telegram bot. Do not delete any progress files.

Open this file using Colab's **File > Upload notebook**. Select a **CPU / None** runtime
for THIS notebook. Do not change the runtime of the original upload while it runs.
You do not need a GPU, a new channel, a new bot, or another copy of the photo archive.


In [ ]:
#@title 1. Prepare the extra batch and verify the first batch is complete
from pathlib import Path
import base64, hashlib, importlib.util, io, json, sys, zipfile
try:
    from google.colab import drive
except ImportError:
    WORKDIR = Path.cwd() / 'Telegram_Movie1_Migration'
else:
    drive.mount('/content/drive')
    WORKDIR = Path('/content/drive/MyDrive/Telegram_Movie1_Migration')
if not WORKDIR.exists():
    raise ValueError('Original migration folder missing. Use the same Google Drive account as the first upload.')
EXTRA_PACKAGE_B64 = '...'
EXPECTED_PACKAGE_SHA256 = 'a705f30063b0baf22365e0995af91de103a5fbf045c5aab0582dceef204f6f78'

raw = base64.b64decode(EXTRA_PACKAGE_B64)
assert hashlib.sha256(raw).hexdigest() == EXPECTED_PACKAGE_SHA256, 'Extra package checksum failed'
EXTRA_DIR = WORKDIR / 'extra_1219'
EXTRA_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as package:
    for item in package.infolist():
        if Path(item.filename).name != item.filename:
            raise ValueError('Unexpected package path')
        content = package.read(item)
        path = EXTRA_DIR / item.filename
        if path.exists() and path.read_bytes() != content:
            raise ValueError('An extra-pass helper differs. Keep your progress and ask for help.')
        if not path.exists():
            path.write_bytes(content)
def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module
extra_pass = load_module('movie1_extra_pass', EXTRA_DIR / 'extra_pass.py')
BASE_MESSAGES, EXTRA_MESSAGES = extra_pass.read_existing_files(WORKDIR)
uploader = load_module('movie1_uploader', WORKDIR / 'uploader.py')
EXTRA_MANIFEST = json.loads((EXTRA_DIR / 'extra_photos.json').read_text(encoding='utf-8'))
uploader.validate_plan(EXTRA_MESSAGES, EXTRA_MANIFEST)
BASE_STATE_PATH = WORKDIR / ('progress_' + str(extra_pass.TARGET_ID) + '.json')
if Path(str(BASE_STATE_PATH) + '.lock').exists():
    raise ValueError('The first upload may still be running. Wait for it to finish; do not delete an active lock.')
if not BASE_STATE_PATH.exists():
    raise ValueError('First-batch progress not found. Finish the original notebook first.')
saved_base = json.loads(BASE_STATE_PATH.read_text(encoding='utf-8'))
base_path, base_state = extra_pass.assert_base_complete(
    uploader, WORKDIR, BASE_MESSAGES, saved_base['bot_id'], extra_pass.TARGET_ID)
EXTRA_STATE_PATH = WORKDIR / ('progress_extra_' + str(extra_pass.TARGET_ID) + '.json')
CACHE = WORKDIR / 'photo_cache'
print('First batch verified: 2573/2573 complete.')
print('Extra photo posts ready:', len(EXTRA_MESSAGES))
print('No Telegram posts sent by this cell.')


## 2. Connect the SAME bot

Enter the same bot token privately. The destination ID is already filled in from your
channel. This cell verifies channel access and checks the bot matches the original
progress file. It does not post anything.


In [ ]:
#@title 2. Connect the same bot to Bollywood Movie
import getpass
token = None
try:
    from google.colab import userdata
    token = userdata.get('TELEGRAM_BOT_TOKEN')
except Exception:
    pass
if not token:
    token = getpass.getpass('Same bot token (hidden): ').strip()
API = uploader.TelegramAPI(token)
del token
BOT, CHANNEL = uploader.validate_target(API, extra_pass.TARGET_ID)
extra_pass.assert_base_complete(uploader, WORKDIR, BASE_MESSAGES, BOT['id'], CHANNEL['id'])
extra_state = uploader.load_state(EXTRA_STATE_PATH, EXTRA_MESSAGES, BOT['id'], CHANNEL['id'])
print('Destination:', CHANNEL['title'], '| ID:', CHANNEL['id'])
print('First batch: 2573/2573 | Extra batch already sent:', len(extra_state['sent']), '/ 1219')
if extra_state.get('pending'):
    print('Resolve this uncertain extra delivery using the recovery cell:', extra_state['pending'])


## 3. Upload or resume the extra 1,219 photo posts

Type the exact phrase shown by the cell. Already-completed extra posts are skipped.
The notebook sends only excluded source IDs, in their original chronological order.
Each photo keeps its complete saved caption and links.

Keep the Colab tab open during a normal interactive session. Closing the ChatGPT tab
does not affect Colab. Colab Pro+ documents background execution with available compute
units; normal sessions should not be relied on to finish after you close Colab.

Three-second pauses alone total about **61 minutes** for this extra batch. Downloads,
uploads, and Telegram's waiting periods add time. Do not run two upload processes.


In [ ]:
#@title 3. Upload / resume the 1,219 extra photo posts
expected = 'UPLOAD EXTRA 1219 TO ' + CHANNEL['title']
answer = input('Type exactly: ' + expected + '\n')
if answer == expected:
    extra_state = extra_pass.run_extra(uploader, API, WORKDIR, BASE_MESSAGES, EXTRA_MESSAGES,
                                       EXTRA_MANIFEST, BOT['id'], CHANNEL['id'], limit=0, interval=3.0)
    total = len(BASE_MESSAGES) + len(extra_state['sent'])
    print('Total original + extra posts completed:', total, '/ 3792')
else:
    print('No posts sent.')


## Recovery — only after ALL upload processes have stopped

Reopen THIS extra notebook to resume the second pass. Do not reset or delete either
progress file. A hard runtime crash may leave locks for the base and extra state.
Remove stale locks only once you have confirmed that no upload process is active.

If a send was interrupted after Telegram may have received it, inspect the destination
channel. Mark it `sent` only if the photo/caption exists, and enter its actual new
Telegram post ID (the last number in a copied post link). Choose `not_sent` only after
checking it is absent. If unsure, stop and ask for help.


In [ ]:
#@title Recovery for an interrupted EXTRA batch (no sends)
locks = [Path(str(BASE_STATE_PATH) + '.lock'), Path(str(EXTRA_STATE_PATH) + '.lock')]
existing_locks = [path for path in locks if path.exists()]
if existing_locks:
    answer = input('After stopping ALL upload processes, type ALL UPLOADS STOPPED to clear stale locks: ')
    if answer != 'ALL UPLOADS STOPPED':
        raise ValueError('Locks retained.')
    for path in existing_locks:
        path.unlink()
if EXTRA_STATE_PATH.exists():
    recovery_state = json.loads(EXTRA_STATE_PATH.read_text(encoding='utf-8'))
    pending = recovery_state.get('pending')
    if pending:
        print('Check this extra post in Telegram:', pending)
        action = input('After checking, type sent or not_sent: ').strip()
        new_id = None
        if action == 'sent':
            new_id = int(input('Actual new Telegram message ID: ').strip())
        if action not in ('sent', 'not_sent'):
            raise ValueError('No delivery status changed.')
        uploader.resolve_pending(EXTRA_STATE_PATH, pending['source_id'], action, new_id)
        print('Recovery saved. Rerun Step 3 to continue.')
    else:
        print('No uncertain delivery. Rerun Step 3 to continue.')
else:
    print('Extra upload has not started. Run Step 2, then Step 3.')


## Details

- The base progress file is read and locked, never rewritten by this second pass.
- The extra progress is `progress_extra_-1004492918137.json` in the same migration folder.
- All 1,219 excluded photo records are restored to the posting queue, including older
  captions/links. No additional quality or duplicate filtering is applied.
- The 29 missing-video records remain excluded, as requested.
- The cache is reused; a file is redownloaded only if missing or its hash differs.
- Original captions and URLs are preserved; link destinations are not independently checked.
- The first-batch completion check, separate resume state, non-overlap, source integrity,
  and base-file preservation were tested locally. Live Telegram sending remains in Colab.

[Colab runtime FAQ](https://research.google.com/colaboratory/faq.html) ·
[Colab background execution](https://colab.research.google.com/notebooks/pro.ipynb)
